# Complete Preprocessing Workflow

**Concept:**
A Complete Preprocessing Workflow is the full, ordered pipeline that takes a raw, messy dataset and converts it into a clean, validated, ML-ready dataset. It stitches together every individual preprocessing technique (missing values, duplicates, outliers, encoding, scaling, imbalance handling, feature selection, splitting) into one coherent, repeatable sequence — instead of treating each technique as an isolated exercise.

**Why it is required / what problem it solves:**
Real-world data is never clean. It has missing values, duplicate rows, wrong data types, outliers, inconsistent categories, imbalanced classes, and irrelevant features. If any single step is skipped or done out of order (e.g., scaling before splitting, causing data leakage), the model trained on it will be biased, inaccurate, or fail in production. A complete workflow solves this by enforcing the correct order of operations, ensures reproducibility, prevents data leakage, and produces a dataset a model can actually be trained on. It also mirrors what happens in real ML projects — data almost never arrives ready to model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE

# ---------- STEP 1: LOAD DATASET (creating our own synthetic business dataset) ----------
np.random.seed(42)
n = 500

data = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "age": np.random.randint(18, 70, n).astype(float),
    "income": np.random.normal(50000, 15000, n),
    "tenure_months": np.random.randint(1, 60, n),
    "monthly_charges": np.random.normal(70, 20, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "contract_type": np.random.choice(["Month-to-Month", "One Year", "Two Year"], n, p=[0.6, 0.25, 0.15]),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "churn": np.random.choice(["Yes", "No"], n, p=[0.2, 0.8])  # imbalanced target
})

# Inject missing values
for col in ["age", "income", "monthly_charges"]:
    data.loc[data.sample(frac=0.05, random_state=1).index, col] = np.nan

# Inject duplicate rows
data = pd.concat([data, data.iloc[0:5]], ignore_index=True)

# Inject outliers
data.loc[10, "income"] = 500000
data.loc[20, "monthly_charges"] = 900

print("STEP 1 - Dataset loaded. Shape:", data.shape)

# ---------- STEP 2: INSPECT DATASET ----------
print("\nSTEP 2 - Inspection:")
print(data.head(3))
print(data.info())

# ---------- STEP 3: IDENTIFY DATA TYPES ----------
numeric_cols = ["age", "income", "tenure_months", "monthly_charges"]
categorical_cols = ["gender", "contract_type", "region"]
print("\nSTEP 3 - Numeric columns:", numeric_cols)
print("STEP 3 - Categorical columns:", categorical_cols)

# ---------- STEP 4: IDENTIFY MISSING VALUES ----------
print("\nSTEP 4 - Missing values:\n", data.isnull().sum()[data.isnull().sum() > 0])

# ---------- STEP 5: HANDLE MISSING VALUES ----------
data["age"] = data["age"].fillna(data["age"].median())
data["income"] = data["income"].fillna(data["income"].median())
data["monthly_charges"] = data["monthly_charges"].fillna(data["monthly_charges"].mean())
print("\nSTEP 5 - Missing values after handling:", data.isnull().sum().sum())

# ---------- STEP 6: DETECT DUPLICATES ----------
print("\nSTEP 6 - Duplicate rows found:", data.duplicated().sum())

# ---------- STEP 7: HANDLE DUPLICATES ----------
data = data.drop_duplicates().reset_index(drop=True)
print("STEP 7 - Shape after removing duplicates:", data.shape)

# ---------- STEP 8: VALIDATE DATA ----------
assert (data["age"] >= 18).all() and (data["age"] <= 100).all(), "Invalid age found"
assert data["tenure_months"].min() >= 0, "Negative tenure found"
print("\nSTEP 8 - Data validation passed (ages and tenure within valid ranges)")

# ---------- STEP 9: DETECT OUTLIERS (IQR method) ----------
def detect_outliers_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)]

print("\nSTEP 9 - Outliers in income:", len(detect_outliers_iqr(data["income"])))
print("STEP 9 - Outliers in monthly_charges:", len(detect_outliers_iqr(data["monthly_charges"])))

# ---------- STEP 10: TREAT OUTLIERS (capping) ----------
def cap_outliers(df, col):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower, upper)
    return df

for col in ["income", "monthly_charges"]:
    data = cap_outliers(data, col)
print("\nSTEP 10 - Outliers capped. Max income now:", round(data["income"].max(), 2))

# ---------- STEP 11 & 12 handled inside pipeline (encoding + scaling) ----------
# ---------- STEP 13: HANDLE CLASS IMBALANCE (after split, using SMOTE) ----------
# ---------- STEP 14: FEATURE SELECTION (after encoding, inside pipeline flow) ----------

# ---------- STEP 15: SPLIT DATA (done BEFORE scaling/encoding fit to avoid leakage) ----------
X = data.drop(columns=["customer_id", "churn"])
y = data["churn"].map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("\nSTEP 15 - Train shape:", X_train.shape, "Test shape:", X_test.shape)

# ---------- STEP 16: BUILD PREPROCESSING PIPELINE (Steps 11 & 12 inside) ----------
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),                                   # STEP 12: Scale
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)          # STEP 11: Encode
])

pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)
print("\nSTEP 16 - Pipeline built. Transformed train shape:", X_train_transformed.shape)

# ---------- STEP 13: HANDLE CLASS IMBALANCE (SMOTE, applied ONLY on training data) ----------
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_transformed, y_train)
print("\nSTEP 13 - Class balance before SMOTE:\n", y_train.value_counts())
print("STEP 13 - Class balance after SMOTE:\n", pd.Series(y_train_bal).value_counts())

# ---------- STEP 14: PERFORM FEATURE SELECTION ----------
selector = SelectKBest(score_func=f_classif, k=5)
X_train_selected = selector.fit_transform(X_train_bal, y_train_bal)
X_test_selected = selector.transform(X_test_transformed)
print("\nSTEP 14 - Features selected (top 5). Final train shape:", X_train_selected.shape)

# ---------- STEP 17: VALIDATE FINAL DATASET ----------
print("\nSTEP 17 - Final ML-ready dataset validation:")
print("Any NaNs in final train set?", np.isnan(X_train_selected).any())
print("Final train shape:", X_train_selected.shape, "| Final test shape:", X_test_selected.shape)
print("Final class balance in train:", pd.Series(y_train_bal).value_counts().to_dict())


STEP 1 - Dataset loaded. Shape: (505, 9)
STEP 4 - Missing values:
age                 25
income              25
monthly_charges     25
STEP 5 - Missing values after handling: 0
STEP 6 - Duplicate rows found: 5
STEP 7 - Shape after removing duplicates: (500, 9)
STEP 8 - Data validation passed (ages and tenure within valid ranges)
STEP 9 - Outliers in income: 1
STEP 9 - Outliers in monthly_charges: 1
STEP 10 - Outliers capped. Max income now: 91452.33
STEP 15 - Train shape: (400, 7) Test shape: (100, 7)
STEP 16 - Pipeline built. Transformed train shape: (400, 13)
STEP 13 - Class balance before SMOTE:
0    320
1     80
STEP 13 - Class balance after SMOTE:
0    320
1    320
STEP 14 - Features selected (top 5). Final train shape: (640, 5)
STEP 17 - Final ML-ready dataset validation:
Any NaNs in final train set? False
Final train shape: (640, 5) | Final test shape: (100, 5)
Final class balance in train: {0: 320, 1: 320}


**What this example tells us (according to the topic):**
This single script demonstrates the correct end-to-end order of a real preprocessing workflow: load → inspect → understand types → find and fix missing values → find and fix duplicates → validate the data's logical correctness → find and treat outliers → **split the data before fitting any transformer** (this is the critical leakage-prevention step) → encode and scale using a pipeline fit only on training data → balance classes only on the training set with SMOTE → select the most relevant features → and finally verify the output is clean, numeric, and balanced. Every step feeds into the next, and the final arrays (`X_train_selected`, `y_train_bal`, `X_test_selected`, `y_test`) are directly usable by any scikit-learn model.

**AI/ML Example:**
A team building a churn-prediction model receives raw CRM export data. Running it through this exact workflow — cleaning missing income values, removing duplicate customer records, capping billing outliers, encoding contract types, scaling numeric features, balancing the rare "churn=Yes" class, and selecting the top predictive features — turns an unusable spreadsheet into a dataset ready to train a classifier like Logistic Regression or XGBoost.

**Business Example:**
A telecom company's customer database has duplicate entries from system merges, missing income fields from incomplete surveys, and a huge class imbalance because only 20% of customers churn. Applying this complete workflow before building a churn-prediction dashboard ensures the business insights ("customers on month-to-month contracts churn 3x more") are based on clean, correctly balanced data rather than noise or duplicated records skewing the numbers.

**AI/ML Engineer Use Case:**
An ML engineer wraps Steps 1–17 into a single reusable `sklearn.Pipeline` + custom cleaning functions, version-controls it, and deploys it as a preprocessing microservice. Every time new raw data arrives (daily batch), it passes through this exact same workflow automatically, guaranteeing that the exact same cleaning, encoding, scaling, and feature-selection logic used during training is applied identically at inference time — preventing training/serving skew, which is one of the most common causes of production ML failures.